# 05c. Qwen3-0.6B with LoRA: scoring on Kaggle

Scoring run for the LLM candidate, executed on Kaggle (two Tesla T4 GPUs). It loads Qwen3-0.6B, adds the LoRA adapter and classification head saved by the training run (`05b_qwen_lora_train_kaggle.ipynb`, attached as an input), checks that the adapter loaded correctly, and scores all 965,524 validation rows. Training and scoring are separate runs because together they would not fit in Kaggle's 12 hour limit.

In [1]:
!pip install -q --no-deps --force-reinstall git+https://github.com/sap6011/bank-guard.git
!pip install -q --no-deps peft
!pip uninstall -y -q torchao

import importlib.metadata as md

for package in ["torch", "transformers", "peft", "accelerate"]:
    print(package, md.version(package))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
torch 2.11.0+cu130
transformers 5.20.2
peft 0.21.2
accelerate 1.15.0


The same packages and versions as the training run: `bankguard` from GitHub, PEFT without its dependencies, and torchao removed because Kaggle's version is older than PEFT requires.

In [2]:
import hashlib
import time
from pathlib import Path

import numpy as np
import torch

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.features import FEATURES, build_features, fit_currency_median
from bankguard.text import to_text

DATA = Path("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")

tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]

X = build_features(tx, accounts, fit_currency_median(train))
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
del tx, X

val_text = to_text(X_val).tolist()
print("val text:", hashlib.md5("\n".join(val_text).encode()).hexdigest())

val text: ed4defe73ab6fe9ee28e95ea9e7f12b3


Features are built from the full transaction table, since each validation row's graph counts depend on earlier transfers. The validation sentences give the same MD5 hash as for every other text model (ed4defe7), so Qwen scores exactly the same inputs.

In [3]:
from peft import PeftModel
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL = "Qwen/Qwen3-0.6B"
MAX_LEN = 192
ADAPTER = next(Path("/kaggle/input").rglob("adapter_config.json")).parent
print("adapter:", ADAPTER)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, dtype=torch.float32)
base.config.pad_token_id = tokenizer.pad_token_id
base.config.use_cache = False
model = PeftModel.from_pretrained(base, ADAPTER).eval()


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i], "label": int(self.labels[i])}


def collate(batch):
    enc = tokenizer(
        [b["text"] for b in batch],
        truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt",
        return_token_type_ids=False,
    )
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc


def laundering_prob(output):
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=1)[:, 1].numpy()


scorer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="/kaggle/temp/qwen-score", per_device_eval_batch_size=64, fp16=True,
        remove_unused_columns=False, report_to="none",
    ),
    data_collator=collate,
)

# Check: every laundering row plus 5,000 normal rows.
pos = np.flatnonzero(y_val == 1)
neg = np.random.default_rng(0).choice(np.flatnonzero(y_val == 0), size=5_000, replace=False)
start = time.time()
p_pos = laundering_prob(scorer.predict(TextDataset([val_text[i] for i in pos], y_val[pos])))
p_neg = laundering_prob(scorer.predict(TextDataset([val_text[i] for i in neg], y_val[neg])))
rate = (len(pos) + len(neg)) / (time.time() - start)
print(f"mean laundering probability: laundering rows {p_pos.mean():.3f}, normal rows {p_neg.mean():.3f}")
print(f"{rate:.0f} rows/s, about {len(val_text) / rate / 60:.0f} min for all validation")

adapter: /kaggle/input/notebooks/saptarshikuila/bankguard-qwen-lora-train/qwen_lora_adapter


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


mean laundering probability: laundering rows 0.287, normal rows 0.001
43 rows/s, about 378 min for all validation


Qwen3-0.6B is loaded with a new two-class head, and the saved adapter supplies the trained LoRA matrices and the head; the `score.weight MISSING` line in the load report is filled by the adapter immediately after. Before the long run, a quick check scored all 1,036 laundering rows and 5,000 random normal rows: the mean laundering probability was 0.287 for laundering rows against 0.001 for normal rows, so the adapter loaded correctly (a randomly initialised head would give both groups about the same value). The probabilities are low overall because training used 1.25% laundering; only the ranking matters for the metrics. Scoring ran at 43 rows per second, about 6.3 hours for all of validation.

In [4]:
import json

from bankguard.evaluate import evaluate

CHUNK = 100_000
parts = []
start = time.time()
for lo in range(0, len(val_text), CHUNK):
    hi = min(lo + CHUNK, len(val_text))
    parts.append(laundering_prob(scorer.predict(TextDataset(val_text[lo:hi], y_val[lo:hi]))))
    np.save("/kaggle/working/qwen_val_scores_partial.npy", np.concatenate(parts))
    print(f"{hi:,} of {len(val_text):,} rows, {(time.time() - start) / 60:.0f} min", flush=True)
score_minutes = (time.time() - start) / 60

val_scores = np.concatenate(parts)
np.save("/kaggle/working/qwen_val_scores.npy", val_scores)
Path("/kaggle/working/qwen_val_scores_partial.npy").unlink()

metrics = evaluate(y_val, val_scores, "Qwen3-0.6B LoRA, same features as text")
metrics.update(score_minutes=score_minutes, adapter=str(ADAPTER))
with open("/kaggle/working/qwen_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

100,000 of 965,524 rows, 39 min


200,000 of 965,524 rows, 77 min


300,000 of 965,524 rows, 116 min


400,000 of 965,524 rows, 155 min


500,000 of 965,524 rows, 194 min


600,000 of 965,524 rows, 233 min


700,000 of 965,524 rows, 272 min


800,000 of 965,524 rows, 311 min


900,000 of 965,524 rows, 349 min


965,524 of 965,524 rows, 374 min
{'model': 'Qwen3-0.6B LoRA, same features as text', 'pr_auc': 0.30601298746380784, 'recall_at_0.1%_fpr': 0.35714285714285715, 'recall_at_1.0%_fpr': 0.7635135135135135, 'score_minutes': 374.4801977356275, 'adapter': '/kaggle/input/notebooks/saptarshikuila/bankguard-qwen-lora-train/qwen_lora_adapter'}


All of validation was scored in ten chunks of 100,000 rows in 374 minutes, saving the scores after each chunk so a late failure would not lose the run. Qwen3-0.6B with LoRA reaches validation PR-AUC 0.306, recall 0.357 at a 0.1% false positive rate and 0.764 at 1%. That is about level with ModernBERT after two epochs (0.302, 0.360, 0.783) despite four times as many parameters, and below LightGBM (0.341, 0.379, 0.764) and ModernBERT after four epochs (0.353). The comparison is discussed in `05a_qwen_lora.ipynb`.